In [ ]:
ingest_date = None

In [ ]:
%run 00_config

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def _staged(resource, fmt):
    tbl = f"bronze_{fmt}_{resource.lower()}"
    if not spark.catalog.tableExists(tbl):
        return None
    spec = FLATTEN[resource]
    b = spark.read.table(tbl)
    if fmt == "json":
        cols = []
        for name, path in spec.items():
            c = F.get_json_object("resource_payload", path)
            if name in REFERENCE_COLS:
                c = F.regexp_replace(c, r"^[A-Za-z]+/", "")
            cols.append(c.alias(name))
    else:                                   # xml bronze already has flat columns
        cols = [F.col(name) for name in spec.keys()]
    return b.select(
        F.col("id").alias("business_id"), "version_id", "last_updated",
        "extraction_timestamp", "run_id", "ingest_date",
        F.lit(fmt).alias("source_format"), *cols)

def build_staged(resource):
    spec = FLATTEN[resource]
    parts = [p for p in (_staged(resource, "json"), _staged(resource, "xml")) if p is not None]
    flat = parts[0]
    for p in parts[1:]:
        flat = flat.unionByName(p)
    # dedup to the latest version per business_id, across BOTH sources
    w = Window.partitionBy("business_id").orderBy(
        F.col("last_updated").desc_nulls_last(), F.col("version_id").desc_nulls_last())
    latest = flat.withColumn("rn", F.row_number().over(w)).where("rn = 1").drop("rn")
    tracked = list(spec.keys())
    return latest.withColumn("record_hash",
        F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in tracked]), 256))

In [ ]:
from delta.tables import DeltaTable

def scd2_merge(resource, staged=None):
    staged = staged if staged is not None else build_staged(resource)
    table = f"silver_{resource.lower()}"
    src = (staged
        .withColumn("effective_from", F.current_timestamp())
        .withColumn("effective_to", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .withColumn("silver_load_timestamp", F.current_timestamp()))
    if not spark.catalog.tableExists(table):
        src.write.format("delta").saveAsTable(table)
        print(f"{table}: initial load {src.count()}")
        return
    tgt = DeltaTable.forName(spark, table)
    current = tgt.toDF().where("is_current = true").select("business_id", F.col("record_hash").alias("cur_hash"))
    changed = src.join(current, "business_id", "left").where("cur_hash IS NULL OR record_hash <> cur_hash")
    inserters = changed.drop("cur_hash").withColumn("mergeKey", F.lit(None).cast("string"))
    closers = changed.where("cur_hash IS NOT NULL").drop("cur_hash").withColumn("mergeKey", F.col("business_id"))
    staged_updates = inserters.unionByName(closers)
    insert_map = {c: f"s.{c}" for c in src.columns}
    (tgt.alias("t").merge(staged_updates.alias("s"), "t.business_id = s.mergeKey AND t.is_current = true")
        .whenMatchedUpdate(set={"is_current": "false", "effective_to": "current_timestamp()"})
        .whenNotMatchedInsert(values=insert_map).execute())
    print(f"{table}: merged {changed.count()}")

In [ ]:
for resource in resources:
    scd2_merge(resource)

In [ ]:
display(spark.read.table("silver_patient")
        .select("business_id", "name_family", "gender", "marital_status",
                "is_current", "effective_from", "effective_to").limit(10))

In [ ]:
import time
pid = (spark.read.table("silver_patient")
       .where("is_current = true AND (marital_status IS NULL OR marital_status NOT LIKE 'DEMO-%')")
       .limit(1).collect()[0]["business_id"])
changed = (build_staged("Patient").where(F.col("business_id") == pid)
           .withColumn("marital_status", F.lit(f"DEMO-{int(time.time())}")))
tracked = list(FLATTEN["Patient"].keys())
changed = changed.withColumn("record_hash",
    F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in tracked]), 256))
scd2_merge("Patient", staged=changed)
display(spark.read.table("silver_patient").where(F.col("business_id") == pid)
        .select("business_id", "marital_status", "is_current", "effective_from", "effective_to")
        .orderBy("effective_from"))